# Project 5: Finance Analytics
**Domain:** Finance  
**Dataset Source:** `data/stock_market.csv`  

---

## Executive Overview
Moving averages, risk metrics, and returns.

---


In [ ]:
import os
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import DataLoader
from src.statistical_analysis import StatisticalAnalyzer
from src.visualization import Visualizer

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')


## 1. Data Quality & Preprocessing
Checklist:
✓ Dataset shape
✓ Missing values
✓ Duplicate rows
✓ Numeric outliers


In [ ]:
loader = DataLoader('../data/stock_market.csv')
df = loader.load_data()

print("==============================")
print("     RAW DATA QUALITY")
print("==============================")
print(loader.generate_data_quality_report())


In [ ]:
df = loader.clean_missing_values({})

print("==============================")
print("   CLEANED DATA QUALITY")
print("==============================")
print(loader.generate_data_quality_report())


### Outlier Analysis
**Business Question:** Are there anomalous trading volumes?


In [ ]:
sns.boxplot(data=df, x='Volume')
plt.show()

**Finding:** High volume days are present.  
**Meaning:** These represent institutional buying/selling or news events.  
**Recommendation:** Monitor these days for volatility.

### Advanced Pandas: Rolling & Shift
**Business Question:** What is the 20-Day Moving Average?


In [ ]:
df['20D_MA'] = df['Close'].rolling(20).mean()
df['DailyReturn_Pct'] = df['Close'].pct_change() * 100
display(df[['Date', 'Close', '20D_MA', 'DailyReturn_Pct']].tail())

### Risk Metrics
**Business Question:** What is the Sharpe Ratio and Max Drawdown?


In [ ]:
stats = StatisticalAnalyzer(df.dropna())
sharpe = stats.calculate_sharpe_ratio('DailyReturn_Pct')
drawdown = stats.calculate_max_drawdown('Close')
print(f'Sharpe Ratio: {sharpe}\nMax Drawdown: {drawdown}%')

**Finding:** Sharpe ratio indicates risk-adjusted return.  
**Meaning:** Provides insight into portfolio quality.  
**Recommendation:** Use these metrics for asset allocation.

### Statistical Hypothesis Testing
**Business Question:** Is volume correlated with volatility (absolute daily return)?


In [ ]:
df['Abs_Return'] = df['DailyReturn_Pct'].abs()
stats = StatisticalAnalyzer(df.dropna())
res = stats.pearson_correlation_test('Volume', 'Abs_Return')
print(stats.format_hypothesis_report(
    'No correlation between volume and volatility.', 'Positive correlation exists.', 'Pearson Correlation', 'r', res['r_statistic'], res['p_value'], 'Days with high trading volume correlate with price swings.', 'The dataset does not provide sufficient evidence of a statistically significant relationship between trading volume and absolute daily returns.', why_it_matters_reject='Suggests risk managers monitor market volume to anticipate risk events.', ci_lower=res['ci_lower'], ci_upper=res['ci_upper']
))

## Limitations
- Dataset size is limited.
- Results are observational.
- Correlation does not imply causation.
- Some variables contain missing observations.
- External factors are not included.
